# Email extraction funnel audit

## tl;dr

The extractor is operational and internally consistent, but the observed lead-to-email rate is not a clean measure of usable email yield. The saved snapshot shows a 4.39% unique-business hit rate, a sharp decline in later cohorts, and a measurable repeated-address contamination risk.

## Context & Methods

This is a read-only diagnostic companion for `email_funnel_audit.sql`. It uses unique Google Maps `place_id` as the business grain and treats `completed` plus `no_email` as processed outcomes. The JSON input is a fixed snapshot of the live PostgreSQL tables, so rerun `python diagnostics/run_email_funnel_audit.py` before refreshing conclusions.

### Key Assumptions

- A saved `lead_emails` row means publicly discovered, not SMTP-verified.
- Reuse of one exact address across many unrelated businesses is a contamination warning, not proof that every assignment is wrong.
- The live Bing probe is a small deterministic sample of recent campaign misses.

## Data

Load the bounded audit snapshot and label the query outputs used below.

In [ ]:
import json
from pathlib import Path

snapshot_path = Path('email_funnel_snapshot.json')
if not snapshot_path.exists():
    snapshot_path = Path('diagnostics/email_funnel_snapshot.json')
snapshot = json.loads(snapshot_path.read_text(encoding='utf-8'))
audit = {item['query_index']: item['rows'] for item in snapshot['audit_queries']}
snapshot['generated_at_utc'], snapshot['source_tables']

## Results

### 1. Recompute the unique-business funnel

In [ ]:
funnel = audit[1][0]
processed = funnel['unique_businesses_processed']
matched = funnel['unique_businesses_with_saved_email']
{
    'eligible_businesses': funnel['eligible_no_website_businesses'],
    'processed_businesses': processed,
    'businesses_with_email': matched,
    'hit_rate_pct': round(100 * matched / processed, 2),
}

### 2. Compare first-seen campaign cohorts

In [ ]:
campaign_yield = audit[4]
[{
    'campaign_id': row['campaign_id'],
    'unique_businesses': row['unique_businesses'],
    'email_hit_rate_pct': row['email_hit_rate_pct'],
} for row in campaign_yield]

### 3. Quantify repeated-address risk

In [ ]:
reuse = audit[6][0]
adjusted = audit[11][0]
{**reuse, **adjusted}

## Takeaways

- Queueing and persistence reconcile cleanly at the unique-business grain.
- The current 4.39% hit rate is real for the implemented acceptance policy, but later search cohorts yield far fewer indexed business footprints.
- The extractor still accepts some directory-owned or shared addresses; deduplication by `(place_id, email)` does not prevent cross-business contamination.
- The next change should improve source ownership and cross-business reuse checks before relaxing recall filters.